# Drive Capacity Distribution — Gold View

## Purpose
Provide an analytical view of Backblaze drive capacity distribution across source periods.

**Source:** `backblaze_data.sot_silver.drive_daily_stats`  
**Target:** `backblaze_data.sot_gold_views.drive_capacity_distribution_vw`  
**Grain:** One row per drive capacity and source period.  
**Layer Purpose:** Gold — Business-facing analytical view.  

**Calculation:**  
Capacity in TB = capacity_bytes / 1,000,000,000,000     
Failure count = Number of observations where failure = 1.

**Scope:**    
The view groups records by exact capacity value and source period. It does not combine different capacity values into arbitrary capacity bands.

## Business Metrics

- Number of drive observations for each capacity.
- Number of distinct drives for each capacity.
- Number of failed drive observations.
- Drive capacity represented in bytes and terabytes.

In [0]:
%sql
-- Purpose: Aggregate drive observations by exact capacity and source period.
CREATE OR REPLACE VIEW backblaze_data.sot_gold_views.drive_capacity_distribution_vw AS
SELECT
    capacity_bytes,
    ROUND(
        capacity_bytes / 1000000000000.0,
        3
    ) AS capacity_tb,
    _period,
    COUNT(*) AS total_drive_observations,
    COUNT(DISTINCT serial_number) AS distinct_drives_observed,
    SUM(
        CASE
            WHEN failure = 1 THEN 1
            ELSE 0
        END
    ) AS failed_drive_observations

FROM backblaze_data.sot_silver.drive_daily_stats
WHERE capacity_bytes IS NOT NULL
GROUP BY
    capacity_bytes,
    _period;

In [0]:
%sql
ALTER VIEW backblaze_data.sot_gold_views.drive_capacity_distribution_vw
SET TAGS (
    'layer' = 'gold',
    'domain' = 'drive_reliability',
    'subject' = 'capacity_analysis',
    'refresh_type' = 'on_query'
);

In [0]:
%sql
COMMENT ON VIEW backblaze_data.sot_gold_views.drive_capacity_distribution_vw IS
'Provides drive capacity distribution by exact capacity value and source period, including observation counts, distinct drive counts, capacity in terabytes, and failure counts.';

In [0]:
%sql
COMMENT ON COLUMN backblaze_data.sot_gold_views.drive_capacity_distribution_vw.capacity_bytes
IS 'Drive capacity in bytes as reported in the source dataset.';

COMMENT ON COLUMN backblaze_data.sot_gold_views.drive_capacity_distribution_vw.capacity_tb
IS 'Drive capacity converted from bytes to decimal terabytes using 1 TB = 1,000,000,000,000 bytes.';

COMMENT ON COLUMN backblaze_data.sot_gold_views.drive_capacity_distribution_vw._period
IS 'Source dataset period associated with the observations.';

COMMENT ON COLUMN backblaze_data.sot_gold_views.drive_capacity_distribution_vw.total_drive_observations
IS 'Total number of drive observations recorded for the capacity and source period.';

COMMENT ON COLUMN backblaze_data.sot_gold_views.drive_capacity_distribution_vw.distinct_drives_observed
IS 'Number of distinct drive serial numbers observed for the capacity and source period.';

COMMENT ON COLUMN backblaze_data.sot_gold_views.drive_capacity_distribution_vw.failed_drive_observations
IS 'Number of drive observations where the failure indicator equals 1.';

In [0]:
%sql
SELECT *
FROM backblaze_data.sot_gold_views.drive_capacity_distribution_vw
ORDER BY capacity_bytes, _period
LIMIT 20;

In [0]:
%sql
DESCRIBE TABLE EXTENDED
backblaze_data.sot_gold_views.drive_capacity_distribution_vw;